# Historical experiment notebook
This notebook preserves its original protocol and embedded source. For offline result reproduction use `scripts/reproduce.py` in the repository. For a new measurement keep the original experiment identity and do not overwrite old results. Archive checks are intentionally strict.


# CPU–GPU Research — Step 4
## Replication and CPU-thread sensitivity

**Last planned measurement increment; Step 5 is the report and public release.**

Use a **fresh T4 VM**, not the VM that produced Step 3. Disconnect/delete the previous Colab runtime before connecting this notebook. A new browser tab or kernel restart alone may retain the same VM. The hardware gate checks boot IDs.

Upload only `step3_20260919T202327166768Z_export.zip`. We retain the frozen models and original **5% + 16 MiB** memory guard. No refitting or model optimization occurs.

This is a **post-hoc selected robustness study**, not another blind workload evaluation. One-thread results challenge a two-thread-calibrated model and are reported separately.

On any failure, stop collection and run **Section 8** to preserve the partial export. Do not silently retry a block or replace slow trials.

**Local validation:** software tests and CPU analysis plumbing were tested; the actual new T4 measurements have not been executed here.

## 1. Install the bundled extension
Keep Colab's CUDA-enabled PyTorch. This cell writes a separate project folder; your earlier notebooks remain unchanged. The embedded source is verified before extraction.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile

SOURCE_ARCHIVE_B64 = "..."
SOURCE_SHA256 = "6a043f25767ea4e32a3c33b07fc49699c58b73271d73d509cfcb3b75e96b6182"
source_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == SOURCE_SHA256
PROJECT = Path.cwd() / ("cpu_gpu_research_step4_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
PROJECT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
    for entry in z.infolist():
        assert not Path(entry.filename).is_absolute() and ".." not in Path(entry.filename).parts
    z.extractall(PROJECT)
os.chdir(PROJECT)
(PROJECT / "step4_source.zip").write_bytes(source_bytes)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-step4.txt"])

# Each benchmark/reference command runs in a separate process. No live model is
# retained in the notebook kernel between tests, validation, and measurement.
def run_command(arguments, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment.update(OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, *arguments], cwd=PROJECT,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
            env=environment)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            result = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if result != 0:
        raise subprocess.CalledProcessError(result, process.args)

print("Project:", PROJECT)
print("Step 4 code installed. No GPU measurements have run.")

## 2. Upload the Step 3 export
Upload **only** `step3_20260919T202327166768Z_export.zip`, unchanged. For a local Jupyter run, supply its path in `LOCAL_INPUT_PATH` instead of using Colab's dialog.

In [ ]:
LOCAL_INPUT_PATH = ""  # Leave blank in Colab.
if LOCAL_INPUT_PATH:
    STEP3_ZIP = Path(LOCAL_INPUT_PATH).expanduser().resolve()
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one file: the Step 3 export ZIP.")
    name, data = next(iter(uploaded.items()))
    incoming = PROJECT / "incoming"
    incoming.mkdir(exist_ok=True)
    STEP3_ZIP = incoming / Path(name).name
    STEP3_ZIP.write_bytes(data)
    del uploaded, data
if not STEP3_ZIP.is_file():
    raise FileNotFoundError(STEP3_ZIP)
print("Input:", STEP3_ZIP.name)

## 3. Run the software tests
All tests should pass. CUDA-specific tests skip only when no GPU is available; the next hardware gate still requires a T4. These software tests are not the full pretrained checks or new performance results.

In [ ]:
run_command(["-m", "pytest", "-q"], PROJECT / "test_output.txt")

## 4. Audit Step 3 and freeze the robustness protocol
The input hash, original source seal, saved predictions/decisions, and 230 original trials are checked. Existing models are copied byte-for-byte—nothing is refitted.

The new protocol has **13 settings × 4 blocks × 5 repetitions = 260 measured generations**. The thread order is **2, 1, 1, 2**.

A separate margin table re-scores Step 3 for interpretation only. It does not change the primary guard or the frozen choices.

In [ ]:
OUT = PROJECT / "runs" / ("step4_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
run_command(["-m", "offload_research.robustness", "prepare", "--step3", str(STEP3_ZIP),
             "--out", str(OUT)], PROJECT / "prepare_log.txt")
shutil.copyfile(PROJECT / "prepare_log.txt", OUT / "prepare_log.txt")
shutil.copyfile(PROJECT / "test_output.txt", OUT / "test_output.txt")
shutil.copyfile(PROJECT / "step4_source.zip", OUT / "step4_source.zip")
for name in ("README_STEP4.md", "TEST_STATUS_STEP4.md", "STEP3_AUDIT.md", "requirements-step4.txt"):
    shutil.copyfile(PROJECT / name, OUT / name)
(OUT / "environment.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
print("Run directory:", OUT)
print("Prepared and frozen. No new GPU inference yet.")

## 5. Check the fresh VM and numerical correctness
The preflight gate rejects the old Step 3 VM and non-T4 accelerators. Software/CPU differences are recorded—not silently corrected by refitting.

There are **13 pretrained reference checks at each thread count**, for **26 total**. Each checks last-position prefill logits plus three teacher-forced cache steps. A numerical failure blocks timing.

In [ ]:
run_command(["-m", "offload_research.collect_robustness", "preflight", "--out", str(OUT)],
            OUT / "hardware_log.txt")
for threads in (2, 1):
    run_command(["-m", "offload_research.collect_robustness", "reference",
                 "--threads", str(threads), "--out", str(OUT)],
                OUT / f"reference_threads_{threads}.log")

## 6. Run the four measurement blocks
The workloads are batch 3 with prompts 64/256 across all five placements, plus the three original batch-1/prompt-128 controls. Output length stays at 32 tokens.

Run these blocks sequentially without another benchmark running. **Do not rerun this cell after a block has started.** Export partial results through Section 8 after interruption/failure. All raw trials, including spikes, are retained.

In [ ]:
for block in range(4):
    run_command(["-m", "offload_research.collect_robustness", "collect",
                 "--block", str(block), "--out", str(OUT)],
                OUT / f"measurement_block_{block}.log")

## 7. Analyze replication and thread sensitivity separately
`session_replication.csv` compares the new **two-thread** run with Step 3. `thread_comparison.csv` compares **one versus two threads within this new VM**. Ratios above one mean the numerator condition was slower.

Do not pool sessions, interpret tiny differences as an optimization win, or treat token-level timings as independent trials. Synthetic allocation budgets are not enforced physical memory caps.

In [ ]:
run_command(["-m", "offload_research.robustness", "analyze", "--out", str(OUT)],
            OUT / "analysis_log.txt")
coverage = json.loads((OUT / "analysis/coverage.json").read_text())
if not coverage.get("complete"):
    raise RuntimeError("This run is incomplete. Preserve it with Section 8; do not replace failed cases.")
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(OUT / "analysis/prediction_metrics.csv")
display(metrics[metrics.target == "generation_ms"])
display(pd.read_csv(OUT / "analysis/memory_metrics.csv"))
display(pd.read_csv(OUT / "analysis/policy_metrics.csv"))
print((OUT / "analysis/report.md").read_text())

## 8. Export the complete or partial checkpoint
Run this cell after success **or after a failure/interruption**, provided Section 4 created `OUT`. The status distinguishes complete results from a partial run. The original Step 3 ZIP is included unchanged, without another redundant working copy.

Share the downloaded **`step4_..._export.zip`**. This is the final planned experimental checkpoint; Step 5 prepares the technical report and public repository.

In [ ]:
if "OUT" not in globals() or not OUT.exists():
    raise RuntimeError("No run directory exists yet. Complete upload and preparation first.")
run_command(["-m", "offload_research.robustness", "export", "--out", str(OUT)],
            PROJECT / "export_log.txt")
archive_path = OUT.parent / (OUT.name + "_export.zip")
status = json.loads((OUT / "export_status.json").read_text())
print("Complete analysis:", status["complete_analysis"])
print("Saved:", archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Open the ZIP in your local file browser.")

## Research boundaries
No new model, quantization, asynchronous overlap, or guard tuning is added. The one-thread condition is an explicit domain-shift diagnostic. Four blocks in one new VM are not four independent sessions; two available VMs are not a broad hardware sample.

References: [PyTorch benchmarking](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html) and [Linux cgroup v2](https://docs.kernel.org/admin-guide/cgroup-v2.html). These support methodology, not the measured numbers.

No Step 4 GPU benchmark results are prefilled in this notebook.